In [0]:
%pip install yfinance
%pip install pandas

In [0]:
import yfinance as yf
import pandas as pd

from datetime import timedelta

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    current_timestamp,
    max as spark_max
)

from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    StringType,
    TimestampType
)

# ==========================================================
# Configuration
# ==========================================================

DATABASE = "finance_intelligence_hub.bronze"
TABLE_NAME = "stock_prices_raw"
FULL_TABLE_NAME = f"{DATABASE}.{TABLE_NAME}"

# ==========================================================
# Get all tickers
# ==========================================================

tickers = (
    spark.table(f"{DATABASE}.companys")
    .select("ticker")
    .distinct()
    .toPandas()["ticker"]
    .dropna()
    .tolist()
)

print(f"Found {len(tickers)} tickers")

# ==========================================================
# Check table exists
# ==========================================================

tables = (
    spark.sql(f"SHOW TABLES IN {DATABASE}")
    .select("tableName")
    .toPandas()["tableName"]
    .tolist()
)

table_exists = TABLE_NAME in tables

# ==========================================================
# Get last loaded date PER TICKER
# ==========================================================

DEFAULT_START_DATE = "2022-01-01"

if table_exists:

    ticker_last_dates = (
        spark.table(FULL_TABLE_NAME)
        .groupBy("ticker")
        .agg(spark_max("date").alias("max_date"))
        .toPandas()
    )

    start_date_by_ticker = {
        row["ticker"].strip().upper(): row["max_date"].strftime("%Y-%m-%d")
        for _, row in ticker_last_dates.iterrows()
        if row["max_date"] is not None
    }

else:

    start_date_by_ticker = {}

print(f"Tickers with existing history: {len(start_date_by_ticker)}")
print(f"New tickers will start from: {DEFAULT_START_DATE}")

# ==========================================================
# Identify missing tickers (in companys, not yet in stock_prices_raw)
# Normalize (strip whitespace, uppercase) so casing/whitespace
# differences don't create false "missing" or duplicate tickers
# ==========================================================

def normalize(t):
    return t.strip().upper()

all_tickers_norm = {normalize(t): t for t in tickers}
loaded_tickers_norm = {normalize(t) for t in start_date_by_ticker.keys()}

missing_norm = set(all_tickers_norm.keys()) - loaded_tickers_norm
existing_norm = set(all_tickers_norm.keys()) & loaded_tickers_norm

pending_tickers = [all_tickers_norm[t] for t in missing_norm]
loaded_tickers = [all_tickers_norm[t] for t in existing_norm]

print("=" * 60)
print(f"Total tickers in companys table : {len(tickers)}")
print(f"Already in stock_prices_raw     : {len(loaded_tickers)}")
print(f"Missing (never loaded)          : {len(pending_tickers)}")
print("=" * 60)
print("Sample of missing tickers:", pending_tickers[:10])

assert len(pending_tickers) + len(loaded_tickers) == len(all_tickers_norm), \
    "Mismatch: pending + loaded should equal total unique tickers"

# ==========================================================
# Prioritize missing tickers first, then already-loaded ones
# ==========================================================

tickers = pending_tickers + loaded_tickers

print(f"Processing order: {len(pending_tickers)} missing tickers first, "
      f"then {len(loaded_tickers)} already-loaded tickers")

# ==========================================================
# Create empty table if first run
# ==========================================================

if not table_exists:

    empty_schema = spark.createDataFrame(
        [],
        StructType([
            StructField("date", DateType(), True),
            StructField("adj_close", StringType(), True),
            StructField("close", StringType(), True),
            StructField("high", StringType(), True),
            StructField("low", StringType(), True),
            StructField("open", StringType(), True),
            StructField("volume", StringType(), True),
            StructField("ticker", StringType(), True),
            StructField("dwh_loaded_at", TimestampType(), True),
        ])
    )

    (
        empty_schema.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
    )

    table_exists = True

    print("Table created.")

# ==========================================================
# Download + Merge one ticker at a time
# ==========================================================

total_inserted = 0

for i, ticker in enumerate(tickers, start=1):

    ticker_start_date = start_date_by_ticker.get(
        normalize(ticker), DEFAULT_START_DATE
    )

    print(f"[{i}/{len(tickers)}] Loading {ticker} from {ticker_start_date}")

    try:

        df = yf.download(
            ticker,
            start=ticker_start_date,
            auto_adjust=False,
            progress=False,
            threads=True
        )

        if df.empty:
            continue

        df.reset_index(inplace=True)

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)

        df.columns = (
            df.columns
            .str.strip()
            .str.replace(" ", "_")
        )

        df = df.rename(
            columns={
                "Date": "date",
                "Adj_Close": "adj_close",
                "Close": "close",
                "High": "high",
                "Low": "low",
                "Open": "open",
                "Volume": "volume"
            }
        )

        # Keep only new dates
        df = df[
            pd.to_datetime(df["date"]) >=
            pd.to_datetime(ticker_start_date)
        ]

        if df.empty:
            continue

        df["ticker"] = ticker

        df = df[
            [
                "date",
                "adj_close",
                "close",
                "high",
                "low",
                "open",
                "volume",
                "ticker"
            ]
        ]

        # Convert all to string (Bronze Layer)

        for c in [
            "adj_close",
            "close",
            "high",
            "low",
            "open",
            "volume"
        ]:

            df[c] = df[c].astype(str)

        spark_df = spark.createDataFrame(df)

        spark_df = (
            spark_df
            .withColumn(
                "date",
                col("date").cast("date")
            )
            .withColumn(
                "adj_close",
                col("adj_close").cast("string")
            )
            .withColumn(
                "close",
                col("close").cast("string")
            )
            .withColumn(
                "high",
                col("high").cast("string")
            )
            .withColumn(
                "low",
                col("low").cast("string")
            )
            .withColumn(
                "open",
                col("open").cast("string")
            )
            .withColumn(
                "volume",
                col("volume").cast("string")
            )
            .withColumn(
                "ticker",
                col("ticker").cast("string")
            )
            .withColumn(
                "dwh_loaded_at",
                current_timestamp()
            )
        )

        spark_df.createOrReplaceTempView(
            "tmp_stock_prices"
        )


        delta_table = DeltaTable.forName(
            spark,
            FULL_TABLE_NAME
        )

        (
            delta_table.alias("target")
            .merge(
                spark.table("tmp_stock_prices").alias("source"),
                """
                target.ticker = source.ticker
                AND
                target.date = source.date
                """
            )
            .whenNotMatchedInsert(
                values={
                    "date": "source.date",
                    "adj_close": "source.adj_close",
                    "close": "source.close",
                    "high": "source.high",
                    "low": "source.low",
                    "open": "source.open",
                    "volume": "source.volume",
                    "ticker": "source.ticker",
                    "dwh_loaded_at": "source.dwh_loaded_at"
                }
            )
            .execute()
        )

        inserted = spark_df.count()

        total_inserted += inserted

        print(
            f"✓ {ticker} processed ({inserted} rows)"
        )

    except Exception as e:

        print(
            f"✗ Error with {ticker}: {e}"
        )

print("=" * 60)
print("Pipeline Finished")
print(f"Processed Tickers : {len(tickers)}")
print(f"Rows Read         : {total_inserted}")
print("=" * 60)